In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/q17_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Cell 2 optimized

# 1. Filter PART to get the relevant keys on GPU
filtered_keys = part.query(
    "P_BRAND == 'Brand#23' & P_CONTAINER == 'MED BOX'"
).P_PARTKEY

# 2. Compute the 20%‐scaled average quantity per partkey on GPU
avg_map = lineitem.groupby("L_PARTKEY")["L_QUANTITY"].mean() * 0.2

# 3. Filter LINEITEM by those keys and map in the precomputed avg
li = lineitem[["L_PARTKEY", "L_QUANTITY", "L_EXTENDEDPRICE"]]
li = (
    li[li.L_PARTKEY.isin(filtered_keys)]
      .assign(avg=lambda df: df.L_PARTKEY.map(avg_map))
)

# 4. Apply the quantity filter and compute the final metric on GPU
filtered = li[li.L_QUANTITY < li.avg]
total = pd.DataFrame({
    "avg_yearly": [filtered.L_EXTENDEDPRICE.sum() / 7.0]
})

CPU times: user 16.6 s, sys: 2.01 s, total: 18.6 s
Wall time: 18.6 s


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q17/rewritten/o4_mini_high_small/checkpoints/post_cell_2_try_1.pickle